# 02c — Ánh xạ train/validation/test dùng chung

Giao thức chính giữ thứ tự nguồn với khoảng đệm 100 dòng tại hai ranh giới; giao thức phụ ngẫu nhiên phân tầng seed 42. Cấu hình đã đặt trước khi chạy mô hình. Chỉ kiểm tra phân chia và số đếm, không fit transformer/model hoặc chọn ngưỡng. Xem `docs/SPLIT_PROTOCOL.md` để đọc giới hạn: thứ tự nguồn không phải thời gian và khoảng đệm không bảo đảm độc lập.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
while not (ROOT / "configs/splits.json").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Mở notebook trong repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from maintenance.splits import export_splits, load_partition
metadata = export_splits(ROOT)
metadata

{'version': 'ai4i-splits-v1',
 'primary': 'ordered-gap100-v1',
 'seed': 42,
 'config_sha256': 'ad0429fc1e74c2c69844b182b58bc07cfa41d2fa377e107de02adb46dd113f35',
 'config_hash_format': "JSON sort_keys=True, separators=(',', ':'), UTF-8",
 'dataset_version': 'sha256:dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e',
 'sklearn_version': '1.9.1',
 'model_fitted': False,
 'files': [{'split_id': 'ordered-gap100-v1',
   'method': 'source_order',
   'gap_rows': 100,
   'path': 'data/processed/splits/ordered-gap100-v1.csv',
   'sha256': 'ef5c2f584e8592bf0ee2aa30f16a1bb995b33bcde77cefe731f73677c67d0965',
   'rows': 10000},
  {'split_id': 'stratified-seed42-v1',
   'method': 'stratified_random',
   'gap_rows': 0,
   'path': 'data/processed/splits/stratified-seed42-v1.csv',
   'sha256': '755eb3f9f4b933e501f5376aa52b20b97979f8f0b4e94e539564712e351988e4',
   'rows': 10000}]}

In [2]:
display(pd.read_csv(ROOT / "reports/tables/splits/summary.csv"))
train = load_partition(ROOT, metadata["primary"], "train")
validation = load_partition(ROOT, metadata["primary"], "validation")
assert train["X"].shape == (6000, 6)
assert validation["X"].shape == (1900, 6)
assert set(train["keys"].reading_id).isdisjoint(validation["keys"].reading_id)
print("Train/validation có sáu đầu vào, tách nhãn và khóa. Chưa nạp test qua loader.")

,split_id,split,rows,failures,min_source_position,max_source_position,failure_rate
0,ordered-gap100-v1,gap,200,3,6001,8100,0.015000
1,ordered-gap100-v1,test,1900,38,8101,10000,0.020000
2,ordered-gap100-v1,train,6000,255,1,6000,0.042500
3,ordered-gap100-v1,validation,1900,43,6101,8000,0.022632
4,stratified-seed42-v1,test,2000,68,2,9985,0.034000
5,stratified-seed42-v1,train,6000,203,1,10000,0.033833
6,stratified-seed42-v1,validation,2000,68,5,9998,0.034000


Train/validation có sáu đầu vào, tách nhãn và khóa. Chưa nạp test qua loader.


## Bàn giao

Ánh xạ local tại data/processed/splits; hash và cấu hình tại data/metadata/splits.json. Không tự chia lại trong mô hình. Chỉ fit trên train và chọn cấu hình/ngưỡng bằng validation. Mở test với allow_test=True sau khi khóa cấu hình; không so sánh test phụ rồi sửa mô hình cho test chính. Các số đếm nhãn toàn nguồn/từng tập đã được xem và phải khai báo trong báo cáo.